# Exploración de las bases de datos — Corporación Favorita

Recorrido tabla por tabla: `train.csv` (completo, filtrado a 2016-2017, todas las tiendas), `transactions.csv`, `stores.csv`. Corre las celdas en orden.

## 0. Configuración inicial

In [ ]:
import pandas as pd
import numpy as np
import os

BASE_DIR = "C:/Tesis"


## 1. train.csv — completo, filtrado a 2016-2017 (todas las tiendas)

`train.csv` tiene ~125 millones de filas (5 GB) repartidas entre 2013 y 2017, así que no lo cargamos entero: lo leemos por partes (*chunks*) y de cada parte nos quedamos con **todas** las filas de 2016 y 2017 (sin muestreo, sin restringir a una tienda). El resultado va a ser grande — varias decenas de millones de filas — así que esto va a tardar unos minutos y el CSV final que guardemos al final va a pesar varios GB. Asegúrate de tener espacio libre en el disco.

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "train.csv")

tamano_gb = os.path.getsize(FILE_PATH) / (1024**3)
print(f"Tamaño en disco: {tamano_gb:.2f} GB")


### Columnas y primeras filas
Esto solo lee un puñado de filas, no carga el archivo completo.

In [ ]:
preview = pd.read_csv(FILE_PATH, nrows=5)
preview


In [ ]:
preview.dtypes


### Carga completa filtrada a 2016-2017
Le especificamos el tipo de dato a cada columna de antemano (`dtype`), para que pandas no tenga que adivinarlo en cada chunk — con un archivo tan grande esto ayuda a que vaya más rápido y use menos memoria. Va imprimiendo el avance para que sepas que sigue corriendo.

In [ ]:
CHUNK_SIZE = 2_000_000
ANIOS = ["2016", "2017"]

DTYPES = {
    "id": "int64",
    "store_nbr": "int16",
    "item_nbr": "int32",
    "unit_sales": "float32",
}

partes = []

for i, chunk in enumerate(pd.read_csv(FILE_PATH, chunksize=CHUNK_SIZE, dtype=DTYPES)):
    filtro_anio = chunk["date"].str[:4].isin(ANIOS)
    partes.append(chunk[filtro_anio])

    if (i + 1) % 10 == 0:
        print(f"Procesadas {(i + 1) * CHUNK_SIZE:,} filas leídas del archivo...")

train_2016_2017 = pd.concat(partes, ignore_index=True)
train_2016_2017["date"] = pd.to_datetime(train_2016_2017["date"])

print(f"\nFilas obtenidas: {len(train_2016_2017):,}")


### Resumen de columnas y tipos

In [ ]:
train_2016_2017.info()


### Primeras filas

In [ ]:
train_2016_2017.head(10)


### Nulos por columna

In [ ]:
train_2016_2017.isna().sum()


### Valores únicos de una columna
`.unique()` te muestra cuáles son los valores distintos; `.value_counts()` además te dice cuántas veces aparece cada uno; `.nunique()` te da solo el total de valores distintos (útil cuando hay muchos, como en `item_nbr`).

In [ ]:
train_2016_2017["onpromotion"].unique()


In [ ]:
train_2016_2017["store_nbr"].value_counts()


In [ ]:
train_2016_2017["item_nbr"].nunique()


### Rango de fechas y variables clave

In [ ]:
print("Rango de fechas:", train_2016_2017["date"].min(), "->", train_2016_2017["date"].max())
print("Tiendas distintas (store_nbr):", train_2016_2017["store_nbr"].nunique())
print("Productos distintos (item_nbr):", train_2016_2017["item_nbr"].nunique())
print("Valores de onpromotion:", train_2016_2017["onpromotion"].unique())


### Estadísticas de unit_sales (la variable a predecir)

In [ ]:
train_2016_2017["unit_sales"].describe()


### Guardar el resultado
Como esta carga tarda harto, conviene guardar el resultado para no tener que repetir el proceso cada vez que quieras trabajar con estos datos — después solo cargas este CSV directo, sin volver a leer los 125 millones de filas de `train.csv`.

In [ ]:
OUTPUT_PATH = os.path.join(BASE_DIR, "train_2016_2017.csv")
train_2016_2017.to_csv(OUTPUT_PATH, index=False)
print("Guardado en:", OUTPUT_PATH)


---
## 2. transactions.csv
Esta tabla es chica (una fila por tienda y día), no necesita muestreo — se carga completa.

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "transactions.csv")
transactions = pd.read_csv(FILE_PATH)
transactions.head()


In [ ]:
transactions.info()


### Tienda con más transacciones
Dos formas de mirarlo: el **total acumulado** por tienda (sumando todos los días), o el **pico** más alto en un solo día.

In [ ]:
# Total acumulado por tienda (de mayor a menor)
transactions.groupby("store_nbr")["transactions"].sum().sort_values(ascending=False)


In [ ]:
tienda_top = transactions.groupby("store_nbr")["transactions"].sum().idxmax()
print("Tienda con más transacciones en total:", tienda_top)


In [ ]:
# Día con más transacciones en una sola tienda (el pico más alto)
transactions.loc[transactions["transactions"].idxmax()]


---
## 3. stores.csv
54 tiendas en total — tabla chica, se carga completa.

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "stores.csv")
stores = pd.read_csv(FILE_PATH)
stores.head()


### Info de la tienda 44

In [ ]:
stores[stores["store_nbr"] == 44].squeeze()


---
## Pendiente
Secciones para `test.csv`, `items.csv`, `oil.csv` y `holidays_events.csv` — todas son tablas chicas (no necesitan muestreo): `pd.read_csv(FILE_PATH)` directo y después `.info()`, `.head()`, `.isna().sum()`, `.describe()` igual que en `stores.csv` / `transactions.csv`.